In [1]:
library(dplyr)
library(lme4)
library(lmerTest)

library(car)
library(effectsize)
library(ggplot2)


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Loading required package: Matrix


Attaching package: 'lmerTest'


The following object is masked from 'package:lme4':

    lmer


The following object is masked from 'package:stats':

    step


Loading required package: carData


Attaching package: 'car'


The following object is masked from 'package:dplyr':

    recode




In [2]:
master_scored_df <- read.csv(
    file = "master_scored_df.csv", 
    header = TRUE,  
    sep = ",", 
    row.names = 1,  
    # stringsAsFactors = FALSE
    )

In [3]:
passing_models_list <- c(
  "anthropic/claude-opus-4-8", 
  "google/gemini-3.1-pro-preview", 
  "google/gemini-3.6-flash", 
  "google/gemma-4-31b", 
  "openai/gpt-5.4-2026-03-05", 
  "openai/gpt-5.5-2026-04-23", 
  "openai/gpt-5.6-sol", 
  "qwen/qwen3-next-80b-a3b-instruct", 
  "xai/grok-4.20-0309-reasoning"
)

In [4]:
clean_scored_df <- master_scored_df %>%
  filter(model %in% passing_models_list) %>%
  filter(is_silent_failure == 0) %>%
  filter(!is.na(response_score), !is.na(inputTokens), !is.na(sample_subset)) %>%
  
  mutate(
    response_score = as.numeric(response_score),
    model = as.factor(model),
    train_structure = as.factor(train_structure),
    sample_subset = as.factor(sample_subset),
    inputTokens_scaled = as.vector(scale(inputTokens))
  )

# Verify preprocessing metrics
cat("[INFO] Preprocessing complete. Clean trials count:", nrow(clean_scored_df), "\n")
cat("[INFO] Evaluated Models (N):", n_distinct(clean_scored_df$model), "\n")
cat("[INFO] Train Structures (N):", n_distinct(clean_scored_df$train_structure), "\n\n")
cat("[INFO] Levels of sample_subset:\n")
print(levels(clean_scored_df$sample_subset))


[INFO] Preprocessing complete. Clean trials count: 6310 
[INFO] Evaluated Models (N): 9 
[INFO] Train Structures (N): 3 

[INFO] Levels of sample_subset:
[1] "reflexivity"  "symmetry"     "transitivity"


In [5]:
# clean_scored_df <- master_scored_df %>%
#   # Exclude silent failures to prevent log-odds distortion
#   filter(is_silent_failure == 0) %>%
#   filter(!is.na(response_score), !is.na(inputTokens)) %>%
#   mutate(
#     response_score = as.numeric(response_score),
#     model = as.factor(model),
#     train_structure = as.factor(train_structure),
#     # Standardize continuous token counts: Mean = 0, SD = 1
#     inputTokens_scaled = as.vector(scale(inputTokens))
#   )

# cat("[INFO] Preprocessing complete. Clean trials count:", nrow(clean_scored_df), "\n")
# cat("[INFO] Evaluated Models (N):", n_distinct(clean_scored_df$model), "\n")
# cat("[INFO] Train Structures (N):", n_distinct(clean_scored_df$train_structure), "\n\n")

## Linear Mixed model

In [6]:
glmm_fit_main_effects <- lme4::glmer(
  formula = response_score ~ inputTokens_scaled + sample_subset + (1 | model) + (1 | train_structure),
  data = clean_scored_df,
  family = binomial(link = "logit"),
  control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5))
)

In [7]:
cat("=========================================================================\n")
cat(" FITTING INTERACTION GLMM: Tokens * Property Type\n")
cat(" Formula: response_score ~ inputTokens_scaled * sample_subset + ...\n")
cat("=========================================================================\n")

glmm_fit_interaction <- lme4::glmer(
  formula = response_score ~ inputTokens_scaled * sample_subset + (1 | model) + (1 | train_structure),
  data = clean_scored_df,
  family = binomial(link = "logit"),
  control = glmerControl(
    optimizer = "bobyqa",
    optCtrl = list(maxfun = 2e5)
  )
)

print(summary(glmm_fit_interaction))

 FITTING INTERACTION GLMM: Tokens * Property Type
 Formula: response_score ~ inputTokens_scaled * sample_subset + ...
Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: response_score ~ inputTokens_scaled * sample_subset + (1 | model) +  
    (1 | train_structure)
   Data: clean_scored_df
Control: glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e+05))

     AIC      BIC   logLik deviance df.resid 
  2922.2   2976.2  -1453.1   2906.2     6302 

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-20.8343   0.0199   0.0622   0.2279   1.8200 

Random effects:
 Groups          Name        Variance Std.Dev.
 model           (Intercept) 10.4655  3.2350  
 train_structure (Intercept)  0.3782  0.6149  
Number of obs: 6310, groups:  model, 9; train_structure, 3

Fixed effects:
                                             Estimate Std. Error z value
(Intercept)                               

In [8]:
cat("\n=================== MODEL COMPARISON (LRT) ===================\n")
# Compare Main Effects model vs. Interaction Model
lrt_result <- anova(glmm_fit_main_effects, glmm_fit_interaction)
print(lrt_result)


=================== MODEL COMPARISON (LRT) ===================
Data: clean_scored_df
Models:
glmm_fit_main_effects: response_score ~ inputTokens_scaled + sample_subset + (1 | model) + (1 | train_structure)
glmm_fit_interaction: response_score ~ inputTokens_scaled * sample_subset + (1 | model) + (1 | train_structure)
                      npar    AIC    BIC  logLik deviance  Chisq Df Pr(>Chisq)
glmm_fit_main_effects    6 2965.9 3006.4 -1477.0   2953.9                     
glmm_fit_interaction     8 2922.2 2976.2 -1453.1   2906.2 47.755  2  4.268e-11
                         
glmm_fit_main_effects    
glmm_fit_interaction  ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1


In [9]:
# cat("=========================================================================\n")
# cat(" FITTING GLMM: response_score ~ inputTokens_scaled + (1|model) + (1|train_structure)\n")
# cat("=========================================================================\n")

# # Using BOBYQA optimizer for bound-constrained optimization
# glmm_fit <- lme4::glmer(
#   formula = response_score ~ inputTokens_scaled + (1 | model) + (1 | train_structure),
#   data = clean_scored_df,
#   family = binomial(link = "logit"),
#   control = glmerControl(
#     optimizer = "bobyqa",
#     optCtrl = list(maxfun = 2e5)
#   )
# )

# print(summary(glmm_fit))

## Tokenization descriptives

In [10]:
# token_desc_stats <- clean_scored_df %>%
#   group_by(model, train_structure, sample_subset) %>%
#   summarise(
#     n = n(),
#     mean_score = mean(response_score, na.rm = TRUE),
#     mean_inputTokens = mean(inputTokens, na.rm = TRUE),
#     sd_inputTokens = sd(inputTokens, na.rm = TRUE),
#     median_inputTokens = median(inputTokens, na.rm = TRUE),
#     # iqr_inputTokens = IQR(inputTokens, na.rm = TRUE),
#     min_inputTokens = min(inputTokens, na.rm = TRUE),
#     max_inputTokens = max(inputTokens, na.rm = TRUE),
#     .groups = "drop"
#   )

In [11]:
# print(token_desc_stats, n = Inf)
# token_desc_stats

In [12]:
token_desc_stats_draft <- clean_scored_df %>%
  group_by(model, train_structure) %>%
  summarise(
    n = n(),
    mean_score = mean(response_score, na.rm = TRUE),
    # mean_inputTokens = mean(inputTokens, na.rm = TRUE),
    median_inputTokens = median(inputTokens, na.rm = TRUE),
    sd_inputTokens = sd(inputTokens, na.rm = TRUE),
    # iqr_inputTokens = IQR(inputTokens, na.rm = TRUE),
    # min_inputTokens = min(inputTokens, na.rm = TRUE),
    # max_inputTokens = max(inputTokens, na.rm = TRUE),
    .groups = "drop"
  )

In [13]:
token_desc_stats_draft

model,train_structure,n,mean_score,median_inputTokens,sd_inputTokens
<fct>,<fct>,<int>,<dbl>,<dbl>,<dbl>
anthropic/claude-opus-4-8,LS,234,0.6581197,882,0.8230743
anthropic/claude-opus-4-8,MTO,234,0.9529915,884,0.6398250
anthropic/claude-opus-4-8,OTM,234,0.9444444,889,1.2291174
google/gemini-3.1-pro-preview,LS,234,1.0000000,615,0.0000000
google/gemini-3.1-pro-preview,MTO,234,1.0000000,602,0.4724150
google/gemini-3.1-pro-preview,OTM,234,1.0000000,612,0.5240974
google/gemini-3.6-flash,LS,234,1.0000000,615,0.0000000
google/gemini-3.6-flash,MTO,234,0.9914530,605,0.6336029
google/gemini-3.6-flash,OTM,234,1.0000000,608,0.6145587


In [14]:
# Fit linear model with interaction term (model * train_structure)
# lm_input_tokens <- lm(inputTokens ~ model * train_structure, data = clean_scored_df)

In [15]:
# anova_results <- car::Anova(lm_input_tokens, type = "II")
# print("--- ANOVA TABLE ---")
# print(anova_results)

In [16]:
# effect_sizes <- effectsize::eta_squared(anova_results, partial = TRUE)
# print("--- EFFECT SIZES (Partial Eta-Squared) ---")
# print(effect_sizes)

In [17]:
# ggplot(clean_scored_df, aes(x = train_structure, y = inputTokens, color = model, group = model)) +
#   stat_summary(fun = mean, geom = "line", linewidth = 0.8) +
#   stat_summary(fun = mean, geom = "point", size = 2) +
#   labs(
#     title = "Input Token Consumption by Model Architecture and Training Structure",
#     x = "Training Structure",
#     y = "Mean Input Tokens",
#     color = "Model Architecture"
#   ) +
#   theme_minimal() +
#   theme(
#     legend.position = "right",
#     panel.grid.minor = element_blank()
#   )